# 5. Sensitivity Analysis and Decision Support

Refit exactly the shared Logistic Regression selected in Notebook 04 and assert score/tier agreement by company-year before generating scenarios. Then export the seven Tableau sources.

Scenarios shift selected raw ratios before applying the unchanged training preprocessing. Margin compression is −2 percentage points; revenue slowdown is −5 points; the combined case adds +5 points to liabilities/assets. Corresponding first differences shift by the same amount. Other features are held fixed: this is a partial-feature sensitivity exercise, not a fully reconciled accounting scenario, forecast or causal estimate.

In [1]:
from pathlib import Path
import sys

# Works from the project root or any folder beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/analysis_helpers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project folder.")
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import numpy as np
import pandas as pd
from analysis_helpers import (read_panel, fit_models, score_table, transform,
                              ranking, TIERS, model_comparison)

panel = read_panel()
bundle = fit_models(panel)
scores = score_table(bundle)
saved = pd.read_csv(ROOT / "data/processed/model_scores.csv", dtype={"cik": str})
check = saved.merge(scores, on=["cik", "feature_year"], suffixes=("_saved", "_current"), validate="one_to_one")
assert len(check) == len(saved) == len(scores)
assert np.allclose(check.predicted_score_saved, check.predicted_score_current, atol=1e-12, rtol=0)
assert check.risk_tier_saved.equals(check.risk_tier_current)
test = bundle["frames"][2]

# Fixed baseline boundary keys retain CIK tie-breaking even at identical scores.
cutoffs = {}
for year, group in ranking(scores).groupby("feature_year"):
    cutoffs[year] = [(-group.iloc[int(np.ceil(len(group) * cap)) - 1].predicted_score,
                     group.iloc[int(np.ceil(len(group) * cap)) - 1].cik) for cap in (0.10, 0.15)]

def sensitivity_tiers(values):
    result = []
    for row, value in zip(scores.itertuples(), values):
        high, review = cutoffs[row.feature_year]
        key = (-value, row.cik)
        result.append(TIERS[0] if key <= high else TIERS[1] if key <= review else TIERS[2])
    return np.array(result)

assert np.array_equal(sensitivity_tiers(scores.predicted_score), scores.risk_tier)
scenarios = {
    "Margin Compression": {"operating_margin": -0.02, "change_in_operating_margin": -0.02},
    "Revenue Slowdown": {"revenue_growth": -0.05, "change_in_revenue_growth": -0.05},
    "Combined Pressure": {"operating_margin": -0.02, "change_in_operating_margin": -0.02,
                          "revenue_growth": -0.05, "change_in_revenue_growth": -0.05,
                          "liabilities_to_assets": 0.05},
}
summary = []
for name, shifts in scenarios.items():
    changed = test.copy()
    for feature, delta in shifts.items():
        changed[feature] = changed[feature] + delta
    values = bundle["lr"].predict_proba(transform(changed, bundle["prep"]))[:, 1]
    summary.append({"scenario": name, "mean_score_change": np.mean(values - scores.predicted_score),
                    "p90_score_change": np.quantile(values - scores.predicted_score, 0.9)})
    if name == "Combined Pressure":
        scores["scenario_score"] = values
print(pd.DataFrame(summary).round(6).to_string(index=False))
scenario_tier = sensitivity_tiers(scores.scenario_score)
moved = scores.risk_tier.eq(TIERS[1]) & (scenario_tier == TIERS[0])
print("\nMedium to High under fixed baseline boundaries:", int(moved.sum()), "company-years")
print(test.loc[moved].groupby("sector").size().to_string())
scores.to_csv(ROOT / "data/processed/model_scores.csv", index=False)

          scenario  mean_score_change  p90_score_change
Margin Compression           0.003537          0.009136
  Revenue Slowdown           0.004920          0.012419
 Combined Pressure           0.007568          0.017697

Medium to High under fixed baseline boundaries: 24 company-years
sector
Consumer / Retail                       1
Energy / Capital-intensive              1
Industrial / Manufacturing             12
Technology / Software / Electronics    10


In [3]:
out = ROOT / "dashboard/data"
out.mkdir(parents=True, exist_ok=True)
known = panel.dropna(subset=["next_year_financial_pressure"]).copy()
known["outcome_year"] = known.fiscal_year + 1
kpi = pd.DataFrame({
    "metric": ["n_companies", "n_company_years", "pressure_rate", "median_operating_margin", "median_ocf_to_assets"],
    "label": ["Companies", "Company-years", "Financial pressure rate", "Median operating margin", "Median OCF / assets"],
    "value": [panel.cik.nunique(), len(panel), known.next_year_financial_pressure.mean(),
              panel.operating_margin.median(), panel.operating_cash_flow_to_assets.median()],
    "scope": ["All retained years", "Feature years 2012–2024", "Known next-year labels only", "All retained years", "All retained years"],
})
kpi["display_value"] = [f"{int(v):,}" if i < 2 else f"{v:.1%}" for i, v in enumerate(kpi.value)]
kpi["sort_order"] = range(1, 6)
kpi.to_csv(out / "page1_kpi.csv", index=False)
heat = known.groupby(["sector", "outcome_year"]).agg(
    pressure_rate=("next_year_financial_pressure", "mean"),
    known_company_years=("cik", "size"), pressure_cases=("next_year_financial_pressure", "sum")).reset_index()
heat["pressure_cases"] = heat.pressure_cases.astype(int)
heat.to_csv(out / "page1_pressure_heatmap.csv", index=False)
trend = panel.groupby(["sector", "fiscal_year"]).agg(
    operating_margin=("operating_margin", "median"), liabilities_to_assets=("liabilities_to_assets", "median"),
    ocf_to_assets=("operating_cash_flow_to_assets", "median"), company_years=("cik", "size")).reset_index()
trend = trend.rename(columns={"fiscal_year": "feature_year"})
trend.to_csv(out / "page1_sector_trend.csv", index=False)
scatter = panel.loc[panel.fiscal_year.between(2021, 2022),
    ["cik", "company_name", "sector", "fiscal_year", "liabilities_to_assets", "operating_cash_flow_to_assets",
     "operating_margin", "log_assets", "next_year_financial_pressure"]].copy()
scatter = scatter.rename(columns={"fiscal_year": "feature_year"})
scatter["outcome_year"] = scatter.feature_year + 1
scatter["outcome_label"] = scatter.next_year_financial_pressure.map({0: "No Pressure", 1: "Pressure"}).fillna("Unknown")
scatter.to_csv(out / "page1_scatter.csv", index=False)
info = panel[["cik", "fiscal_year", "company_name", "sector", "operating_margin",
              "operating_cash_flow_to_assets", "liabilities_to_assets", "revenue_growth"]].rename(columns={"fiscal_year": "feature_year"})
details = scores.merge(info, on=["cik", "feature_year"], validate="one_to_one")
details["outcome_year"] = details.feature_year + 1
details["scenario_tier"] = scenario_tier
details["review_rank"] = ranking(details).groupby("feature_year").cumcount().add(1).reindex(details.index)
details.to_csv(out / "page2_risk_scores.csv", index=False)
comparison = model_comparison(bundle)
comparison.loc[comparison.model.ne("Majority")].to_csv(out / "page2_model_comparison.csv", index=False)
tiers = details.groupby(["feature_year", "risk_tier"]).size().rename("n_company_years").reset_index()
tiers["share"] = tiers.n_company_years / tiers.groupby("feature_year").n_company_years.transform("sum")
tiers["tier_order"] = tiers.risk_tier.map({name: i + 1 for i, name in enumerate(TIERS)})
tiers.to_csv(out / "page2_tier_distribution.csv", index=False)
print("\nSeven Tableau CSVs exported:")
for file in sorted(out.glob("*.csv")):
    print(file.name, len(pd.read_csv(file)), "rows")
print("\nGlobal KPIs:")
print(kpi[["label", "display_value", "scope"]].to_string(index=False))
print("\nFinal model comparison:")
print(comparison.round(6).to_string(index=False))


Seven Tableau CSVs exported:
page1_kpi.csv 5 rows
page1_pressure_heatmap.csv 48 rows
page1_scatter.csv 3823 rows
page1_sector_trend.csv 52 rows
page2_model_comparison.csv 3 rows
page2_risk_scores.csv 3305 rows
page2_tier_distribution.csv 6 rows

Global KPIs:
                  label display_value                       scope
              Companies         2,767          All retained years
          Company-years        21,990     Feature years 2012–2024
Financial pressure rate         13.1% Known next-year labels only
Median operating margin          6.7%          All retained years
    Median OCF / assets          8.2%          All retained years

Final model comparison:
              model  accuracy  precision   recall       f1  roc_auc   pr_auc  recall_at_10  precision_at_10  reviewed_at_10  recall_at_15  precision_at_15  reviewed_at_15  recall_at_20  precision_at_20  reviewed_at_20
           Majority  0.830560   0.000000 0.000000 0.000000 0.500000 0.169440      0.037500         0.

## Decision use

Sensitivity tiers use fixed baseline annual boundary keys, including the CIK tie-break. Their workload can exceed 15%; they show movement across the original screening boundary. The operational baseline queue retains the annual capacity rule.

KPI cards summarize the full panel. Heatmap denominators include only known labels and use outcome year. Margin trends use feature year. Scatter observations include unknown labels explicitly. Model and tier metrics count company-years, so the same company may appear in both test years. Financial ratios in queue tooltips come from the same company-year as the score.

Outputs: `model_scores.csv` with combined sensitivity scores and exactly seven CSVs in `dashboard/data/`.